In [1]:
import sys, torch, dgl, numpy as np

print(sys.version)
print("Torch:", torch.__version__)
print("DGL:", dgl.__version__)
print("NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())


3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Torch: 2.2.2+cu121
DGL: 2.0.0
NumPy: 1.26.4
CUDA available: False


In [2]:
import os
os.getcwd()


'C:\\Users\\Prabu\\Downloads'

In [1]:
%cd C:/Users/Prabu/Downloads/katabatic1


C:\Users\Prabu\Downloads\katabatic1


C:\Users\Prabu\anaconda3\envs\goggle310\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)


Project root: C:\Users\Prabu\Downloads


In [3]:
import sys
from pathlib import Path

# Change this if your path differs
ROOT = Path("C:/Users/Prabu/Downloads/katabatic1").resolve()
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)


Project root: C:\Users\Prabu\Downloads\katabatic1


In [ ]:

import time
import torch
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel


ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")

RAW_CSV    = ROOT / "raw_data" / "car.csv"
SAMPLE_DIR = ROOT / "sample_data" / "car"
SYNTH_DIR  = ROOT / "synthetic" / "car" / "goggle"
RESULTS_DIR = ROOT / "Results" / "car"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()



pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="car",
        input_dim=6,
        decoder_arch="gcn",
        iter_opt=True
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col=6
)




model = GoggleModel(ds_name="car", input_dim=6)

model.model.load_state_dict(
    torch.load(ROOT / "tmp" / "car.pt", map_location=model.device)
)
model.model.eval()



x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")

with torch.no_grad():
    x_synth = model.sample(x_test)

x_synth.to_csv(SYNTH_DIR / "x_synth_raw.csv", index=False)
y_test.iloc[: len(x_synth)].to_csv(SYNTH_DIR / "y_synth.csv", index=False)


encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# Fit ONLY on real training data
encoder.fit(x_train)

# Encode real data
x_train_enc = pd.DataFrame(encoder.transform(x_train), columns=x_train.columns)
x_test_enc  = pd.DataFrame(encoder.transform(x_test), columns=x_test.columns)

# Fix synthetic categories BEFORE encoding
x_synth_fixed = x_synth.copy()
for col in x_train.columns:
    valid_vals = x_train[col].unique().tolist()
    x_synth_fixed[col] = x_synth_fixed[col].apply(
        lambda x: min(valid_vals, key=lambda v: abs(hash(str(v)) - hash(str(x))))
    )

x_synth_enc = pd.DataFrame(
    encoder.transform(x_synth_fixed),
    columns=x_synth.columns
)

# Save encoded files
x_train_enc.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
x_test_enc.to_csv(SAMPLE_DIR / "x_test.csv", index=False)
x_synth_enc.to_csv(SYNTH_DIR / "x_synth.csv", index=False)




tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()
print(results)

# Save results
pd.DataFrame([results]).to_csv(
    RESULTS_DIR / "goggle_tstr.csv", index=False
)

print("Total runtime (minutes):", round((time.time() - t0) / 60, 2))


Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
Training GOGGLE on (1382, 6)
[Epoch 100/1000] Train 4.4275 | Val 4.4224
[Epoch 200/1000] Train 4.2708 | Val 4.2503
[Epoch 300/1000] Train 4.0502 | Val 3.9666
[Epoch 400/1000] Train 3.5408 | Val 3.3291
Early stopping at epoch 480

Results saved to: Results\car\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6994
F1: 0.2058
AUC: 0.4930

MLP:
Accuracy: 0.5636
F1: 0.2210
AUC: 0.4403

RF:
Accuracy: 0.6214
F1: 0.2245
AUC: 0.3812

XGBoost:
Accuracy: 0.5665
F1: 0.2270
AUC: 0.3869
{'LR': {'Accuracy': 0.6994219653179191, 'F1': 0.205782312925170

In [ ]:

import time
import torch
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel


ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")

RAW_CSV    = ROOT / "raw_data" / "car.csv"
SAMPLE_DIR = ROOT / "sample_data" / "car"
SYNTH_DIR  = ROOT / "synthetic" / "car" / "goggle"
RESULTS_DIR = ROOT / "Results" / "car"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="car",
        input_dim=6,
        decoder_arch="gcn",
        iter_opt=True
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col=6
)


model = GoggleModel(ds_name="car", input_dim=6)

model.model.load_state_dict(
    torch.load(ROOT / "tmp" / "car.pt", map_location=model.device)
)
model.model.eval()


x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")

with torch.no_grad():
    x_synth = model.sample(x_test)

x_synth.to_csv(SYNTH_DIR / "x_synth_raw.csv", index=False)
y_test.iloc[: len(x_synth)].to_csv(SYNTH_DIR / "y_synth.csv", index=False)



encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


encoder.fit(x_train)


x_train_enc = pd.DataFrame(encoder.transform(x_train), columns=x_train.columns)
x_test_enc  = pd.DataFrame(encoder.transform(x_test), columns=x_test.columns)


x_synth_fixed = x_synth.copy()
for col in x_train.columns:
    valid_vals = x_train[col].unique().tolist()
    x_synth_fixed[col] = x_synth_fixed[col].apply(
        lambda x: min(valid_vals, key=lambda v: abs(hash(str(v)) - hash(str(x))))
    )

x_synth_enc = pd.DataFrame(
    encoder.transform(x_synth_fixed),
    columns=x_synth.columns
)


x_train_enc.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
x_test_enc.to_csv(SAMPLE_DIR / "x_test.csv", index=False)
x_synth_enc.to_csv(SYNTH_DIR / "x_synth.csv", index=False)




tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

raw_results = tstr.evaluate()


results = {}

for model_name, metrics in raw_results.items():
    results[model_name] = {
        "Accuracy": metrics["Accuracy"],
        "F1": metrics["F1"],
    }


print("\nTSTR Evaluation Results (CAR – AUC removed):\n")
for model, scores in results.items():
    print(f"{model}:")
    for k, v in scores.items():
        print(f"  {k}: {v:.4f}")

print("\nTotal runtime (minutes):", round((time.time() - t0) / 60, 2))



Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
Training GOGGLE on (1382, 6)
[Epoch 100/1000] Train 4.4275 | Val 4.4224
[Epoch 200/1000] Train 4.2708 | Val 4.2503
[Epoch 300/1000] Train 4.0502 | Val 3.9666
[Epoch 400/1000] Train 3.5408 | Val 3.3291
Early stopping at epoch 480

Results saved to: Results\car\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6994
F1: 0.2058
AUC: 0.4930

MLP:
Accuracy: 0.5636
F1: 0.2210
AUC: 0.4403

RF:
Accuracy: 0.6214
F1: 0.2245
AUC: 0.3812

XGBoost:
Accuracy: 0.5665
F1: 0.2270
AUC: 0.3869

TSTR Evaluation Results (CAR – AUC removed):

LR:
  Accuracy: 

In [ ]:
import time
import torch
import pandas as pd
from pathlib import Path

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel


ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")

RAW_CSV    = ROOT / "raw_data" / "magic.csv"
SAMPLE_DIR = ROOT / "sample_data" / "magic"
SYNTH_DIR  = ROOT / "synthetic" / "magic" / "goggle"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()


pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="magic",
        input_dim=10,         
        decoder_arch="gcn",
        iter_opt=True
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="class"        
)


model = GoggleModel(
    ds_name="magic",
    input_dim=10
)

model.model.load_state_dict(
    torch.load("tmp/magic.pt", map_location=model.device)
)
model.model.eval()


x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")


x_synth = model.sample(x_test)

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_test.iloc[: len(x_synth)].to_csv(
    SYNTH_DIR / "y_synth.csv", index=False
)

print("Synthetic data generated and saved successfully")


tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print("\nTSTR Evaluation Results (MAGIC – AUC included):\n")
for model_name, scores in results.items():
    print(f"{model_name}:")
    print(f"  Accuracy: {scores['Accuracy']:.4f}")
    print(f"  F1:       {scores['F1']:.4f}")
    print(f"  AUC:      {scores['AUC']:.4f}")

print("\nTotal runtime (minutes):", round((time.time() - t0) / 60, 2))


Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
Training GOGGLE on (15216, 10)
[Epoch 100/1000] Train 4.0392 | Val 3.9724
[Epoch 200/1000] Train 3.6699 | Val 3.6028
Early stopping at epoch 297
Synthetic data generated and saved successfully

Results saved to: Results\magic\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6325
F1: 0.4913
AUC: 0.4763

MLP:
Accuracy: 0.6441
F1: 0.5769
AUC: 0.5889

RF:
Accuracy: 0.6309
F1: 0.4543
AUC: 0.4976

XGBoost:
Accuracy: 0.5318
F1: 0.4479
AUC: 0.4189

TSTR Evaluation Results (MAGIC – AUC included):

LR:
  Accuracy: 0.6325
  F1:       0.4913
  AUC:      0.4763
MLP:
  Accuracy: 0.6441
  F1:       0.5769
  AUC:     

In [ ]:
import time
import torch
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel


ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")

RAW_CSV    = ROOT / "raw_data" / "nursery.csv"
SAMPLE_DIR = ROOT / "sample_data" / "nursery"
SYNTH_DIR  = ROOT / "synthetic" / "nursery" / "goggle"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()


pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="nursery",
        input_dim=8,          
        decoder_arch="gcn",
        iter_opt=True
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col=8              
)


model = GoggleModel(
    ds_name="nursery",
    input_dim=8
)

model.model.load_state_dict(
    torch.load("tmp/nursery.pt", map_location=model.device)
)
model.model.eval()


x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")


x_synth = model.sample(x_test)

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_test.iloc[: len(x_synth)].to_csv(
    SYNTH_DIR / "y_synth.csv", index=False
)

print("Synthetic data generated and saved successfully")


from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


x_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test  = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")


x_train = x_train.fillna("MISSING").astype(str)
x_test  = x_test.fillna("MISSING").astype(str)
x_synth = x_synth.fillna("MISSING").astype(str)

x_train_enc = pd.DataFrame(
    encoder.fit_transform(x_train),
    columns=x_train.columns
)

x_test_enc = pd.DataFrame(
    encoder.transform(x_test),
    columns=x_test.columns
)

x_synth_enc = pd.DataFrame(
    encoder.transform(x_synth),
    columns=x_synth.columns
)


x_train_enc.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
x_test_enc.to_csv(SAMPLE_DIR / "x_test.csv", index=False)
x_synth_enc.to_csv(SYNTH_DIR / "x_synth.csv", index=False)


tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

raw_results = tstr.evaluate()


results = {
    model: {
        "Accuracy": metrics["Accuracy"],
        "F1": metrics["F1"],
    }
    for model, metrics in raw_results.items()
}

print("\nTSTR Evaluation Results (NURSERY – encoded, AUC removed):\n")
for model, scores in results.items():
    for k, v in scores.items():
        print(f"{model} {k}: {v:.4f}")

print("\nTotal runtime (minutes):", round((time.time() - t0) / 60, 2))


Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
Training GOGGLE on (10368, 8)
[Epoch 100/1000] Train 2.2643 | Val 2.2462
[Epoch 200/1000] Train 2.2271 | Val 2.2187
Early stopping at epoch 236
Synthetic data generated and saved successfully

Results saved to: Results\nursery\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.3333
F1: 0.1250
AUC: 0.5000

MLP:
Accuracy: 0.3503
F1: 0.1795
AUC: 0.5502

RF:
Accuracy: 0.3333
F1: 0.1250
AUC: 0.5000

XGBoost:
Accuracy: 0.3333
F1: 0.1250
AUC: 0.5000

TSTR Evaluation Results (

In [ ]:
import time
import torch
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel


ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")

(ROOT / "tmp").mkdir(parents=True, exist_ok=True)

RAW_CSV    = ROOT / "raw_data" / "adult.csv"
SAMPLE_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR  = ROOT / "synthetic" / "adult" / "goggle"
RESULTS_DIR = ROOT / "Results" / "adult"


t0 = time.time()


pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="adult",
        input_dim=14,         
        decoder_arch="gcn",
        iter_opt=True
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="class"       
)


model = GoggleModel(
    ds_name="adult",
    input_dim=14
)

ckpt_path = Path("tmp/adult.pt")

if ckpt_path.exists():
    model.model.load_state_dict(
        torch.load(ckpt_path, map_location=model.device)
    )
    print("Loaded checkpoint: tmp/adult.pt")
else:
    print("No checkpoint file found — using in-memory trained model.")

model.model.eval()


x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")


x_synth = model.sample(x_test)

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_test.iloc[: len(x_synth)].to_csv(
    SYNTH_DIR / "y_synth.csv", index=False
)

print("Synthetic data generated and saved successfully")


encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


x_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test  = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_train = x_train.fillna("MISSING").astype(str)
x_test  = x_test.fillna("MISSING").astype(str)
x_synth = x_synth.fillna("MISSING").astype(str)

x_train_enc = pd.DataFrame(
    encoder.fit_transform(x_train),
    columns=x_train.columns
)

x_test_enc = pd.DataFrame(
    encoder.transform(x_test),
    columns=x_test.columns
)

x_synth_enc = pd.DataFrame(
    encoder.transform(x_synth),
    columns=x_synth.columns
)


x_train_enc.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
x_test_enc.to_csv(SAMPLE_DIR / "x_test.csv", index=False)
x_synth_enc.to_csv(SYNTH_DIR / "x_synth.csv", index=False)


tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print("\nTSTR Evaluation Results (ADULT – AUC included):\n")
for model, scores in results.items():
    print(f"{model}:")
    print(f"  Accuracy: {scores['Accuracy']:.4f}")
    print(f"  F1:       {scores['F1']:.4f}")
    print(f"  AUC:      {scores['AUC']:.4f}")

print("\nTotal runtime (minutes):", round((time.time() - t0) / 60, 2))


Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Training GOGGLE on (26048, 14)
 No checkpoint found — using last epoch weights.
No checkpoint file found — using in-memory trained model.
Synthetic data generated and saved successfully

Results saved to: Results\adult\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

MLP:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.3832

RF:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

XGBoost:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

TSTR Evaluation Results (ADULT – AUC included):

LR:
  Accuracy: 0.7593
  F1:       0.4316
  AUC:      0.5000
MLP:
  Accuracy: 0.7593
  F1:       0.4316
  

In [ ]:
import time
import torch
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel


ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")


(ROOT / "tmp").mkdir(parents=True, exist_ok=True)

RAW_CSV    = ROOT / "raw_data" / "adult.csv"
SAMPLE_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR  = ROOT / "synthetic" / "adult" / "goggle"
RESULTS_DIR = ROOT / "Results" / "adult"


t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="adult",
        input_dim=14,         
        decoder_arch="gcn",
        iter_opt=True
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="class"       
)


model = GoggleModel(
    ds_name="adult",
    input_dim=14
)

ckpt_path = Path("tmp/adult.pt")

if ckpt_path.exists():
    model.model.load_state_dict(
        torch.load(ckpt_path, map_location=model.device)
    )
    print("Loaded checkpoint: tmp/adult.pt")
else:
    print("No checkpoint file found — using in-memory trained model.")

model.model.eval()

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")


x_synth = model.sample(x_test)
x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)


y_train = pd.read_csv(SAMPLE_DIR / "y_train.csv")

y_synth = y_train.sample(
    n=len(x_synth),
    replace=True,
    random_state=42
).reset_index(drop=True)

y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Synthetic X and Y generated and saved successfully")


encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


x_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test  = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_train = x_train.fillna("MISSING").astype(str)
x_test  = x_test.fillna("MISSING").astype(str)
x_synth = x_synth.fillna("MISSING").astype(str)


x_train_enc = pd.DataFrame(
    encoder.fit_transform(x_train),
    columns=x_train.columns
)

x_test_enc = pd.DataFrame(
    encoder.transform(x_test),
    columns=x_test.columns
)

x_synth_enc = pd.DataFrame(
    encoder.transform(x_synth),
    columns=x_synth.columns
)


x_train_enc.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
x_test_enc.to_csv(SAMPLE_DIR / "x_test.csv", index=False)
x_synth_enc.to_csv(SYNTH_DIR / "x_synth.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print("\nTSTR Evaluation Results (ADULT – AUC included):\n")
for model, scores in results.items():
    print(f"{model}:")
    print(f"  Accuracy: {scores['Accuracy']:.4f}")
    print(f"  F1:       {scores['F1']:.4f}")
    print(f"  AUC:      {scores['AUC']:.4f}")

print("\nTotal runtime (minutes):", round((time.time() - t0) / 60, 2))


Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Training GOGGLE on (26048, 14)
 No checkpoint found — using last epoch weights.
No checkpoint file found — using in-memory trained model.
Synthetic X and Y generated and saved successfully

Results saved to: Results\adult\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

MLP:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.3758

RF:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

XGBoost:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

TSTR Evaluation Results (ADULT – AUC included):

LR:
  Accuracy: 0.7593
  F1:       0.4316
  AUC:      0.5000
MLP:
  Accuracy: 0.7593
  F1:       0.4316

In [ ]:
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel

ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")

(ROOT / "tmp").mkdir(parents=True, exist_ok=True)

RAW_CSV = ROOT / "raw_data" / "adult.csv"
SAMPLE_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR = ROOT / "synthetic" / "adult" / "goggle"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="adult",
        input_dim=14,
        decoder_arch="gcn",
        iter_opt=True
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="class"
)

model = GoggleModel(ds_name="adult", input_dim=14)

ckpt_path = Path("tmp/adult.pt")
if ckpt_path.exists():
    model.model.load_state_dict(
        torch.load(ckpt_path, map_location=model.device)
    )
    print("Loaded checkpoint: tmp/adult.pt")
else:
    print("No checkpoint found — using in-memory trained model")

model.model.eval()

x_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
y_train = pd.read_csv(SAMPLE_DIR / "y_train.csv")

x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")

x_synth = model.sample(x_test)
x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_train_enc = encoder.fit_transform(
    x_train.fillna("MISSING").astype(str)
)
x_test_enc = encoder.transform(
    x_test.fillna("MISSING").astype(str)
)
x_synth_enc = encoder.transform(
    x_synth.fillna("MISSING").astype(str)
)

pd.DataFrame(x_train_enc, columns=x_train.columns).to_csv(
    SAMPLE_DIR / "x_train.csv", index=False
)
pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    SAMPLE_DIR / "x_test.csv", index=False
)
pd.DataFrame(x_synth_enc, columns=x_train.columns).to_csv(
    SYNTH_DIR / "x_synth.csv", index=False
)

teacher = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

teacher.fit(x_train_enc, y_train.values.ravel())

proba = teacher.predict_proba(x_synth_enc)
classes = teacher.classes_

rng = np.random.default_rng(42)
y_synth = [rng.choice(classes, p=p) for p in proba]

y_synth = pd.DataFrame(y_synth, columns=["class"])

print("Synthetic label distribution:")
print(y_synth["class"].value_counts(normalize=True))

y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print("\nTSTR Evaluation Results (ADULT – FINAL):\n")
for model_name, scores in results.items():
    print(f"{model_name}:")
    print(f"  Accuracy: {scores['Accuracy']:.4f}")
    print(f"  F1:       {scores['F1']:.4f}")
    print(f"  AUC:      {scores['AUC']:.4f}")

print("\nTotal runtime (minutes):", round((time.time() - t0) / 60, 2))


Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Training GOGGLE on (26048, 14)
 No checkpoint found — using last epoch weights.
No checkpoint found — using in-memory trained model
Synthetic label distribution:
class
>50K     0.561799
<=50K    0.438201
Name: proportion, dtype: float64

Results saved to: Results\adult\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.2407
F1: 0.1940
AUC: 0.5000

MLP:
Accuracy: 0.2407
F1: 0.1940
AUC: 0.5099

RF:
Accuracy: 0.2407
F1: 0.1940
AUC: 0.5000

XGBoost:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

TSTR Evaluation Results (ADULT – FINAL):

LR:
  Accuracy: 0.2407
  F1:       0.1940
  AUC:      0.5000
M

In [ ]:


import time
import torch
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel


ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")

RAW_CSV    = ROOT / "raw_data" / "adult.csv"
SAMPLE_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR  = ROOT / "synthetic" / "adult" / "goggle"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / "tmp").mkdir(exist_ok=True)

t0 = time.time()


pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="adult",
        input_dim=14,        
        decoder_arch="gcn",
        iter_opt=True,

        
        epochs=1000,
        patience=999,        
        logging=100,         
        batch_size=256,
        learning_rate=1e-3
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col="class"       
)


model = GoggleModel(
    ds_name="adult",
    input_dim=14
)

ckpt_path = ROOT / "tmp" / "adult.pt"

if ckpt_path.exists():
    model.model.load_state_dict(
        torch.load(ckpt_path, map_location=model.device)
    )
    print("Loaded checkpoint:", ckpt_path)
else:
    print("No checkpoint found — using in-memory trained model")

model.model.eval()


x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")


x_synth = model.sample(x_test)

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_test.iloc[:len(x_synth)].to_csv(
    SYNTH_DIR / "y_synth.csv", index=False
)

print("Synthetic data generated and saved successfully")


encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_train = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test  = pd.read_csv(SAMPLE_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

# force categorical safety
x_train = x_train.fillna("MISSING").astype(str)
x_test  = x_test.fillna("MISSING").astype(str)
x_synth = x_synth.fillna("MISSING").astype(str)

# fit on REAL TRAIN only
x_train_enc = pd.DataFrame(
    encoder.fit_transform(x_train),
    columns=x_train.columns
)

x_test_enc = pd.DataFrame(
    encoder.transform(x_test),
    columns=x_test.columns
)

x_synth_enc = pd.DataFrame(
    encoder.transform(x_synth),
    columns=x_synth.columns
)

x_train_enc.to_csv(SAMPLE_DIR / "x_train.csv", index=False)
x_test_enc.to_csv(SAMPLE_DIR / "x_test.csv", index=False)
x_synth_enc.to_csv(SYNTH_DIR / "x_synth.csv", index=False)


tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

print("\nTSTR Evaluation Results (ADULT – FORCED TRAINING):\n")
for model_name, scores in results.items():
    print(f"{model_name}:")
    print(f"  Accuracy: {scores['Accuracy']:.4f}")
    print(f"  F1:       {scores['F1']:.4f}")
    print(f"  AUC:      {scores['AUC']:.4f}")

print("\nTotal runtime (minutes):", round((time.time() - t0) / 60, 2))


Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Training GOGGLE on (26048, 14)
 No checkpoint found — using last epoch weights.
No checkpoint found — using in-memory trained model
Synthetic data generated and saved successfully

Results saved to: Results\adult\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

MLP:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.3832

RF:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

XGBoost:
Accuracy: 0.7593
F1: 0.4316
AUC: 0.5000

TSTR Evaluation Results (ADULT – FORCED TRAINING):

LR:
  Accuracy: 0.7593
  F1:       0.4316
  AUC:      0.5000
MLP:
  Accuracy: 0.7593
  F1:       0.4316
  AUC

In [ ]:


import time
import torch
import pandas as pd
from pathlib import Path

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.evaluate.tstr.evaluation import TSTREvaluation
from katabatic.models.goggle import GoggleModel


ROOT = Path("C:/Users/Prabu/Downloads/katabatic1")

RAW_CSV    = ROOT / "raw_data" / "shuttle.csv"
SAMPLE_DIR = ROOT / "sample_data" / "shuttle"
SYNTH_DIR  = ROOT / "synthetic" / "shuttle" / "goggle"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / "tmp").mkdir(exist_ok=True)

t0 = time.time()


pipeline = TrainTestSplitPipeline(
    model=lambda: GoggleModel(
        ds_name="shuttle",
        input_dim=9,          
        decoder_arch="gcn",
        iter_opt=True,

    
        epochs=1000,
        patience=999,         
        logging=100,          
        batch_size=512,      
        learning_rate=1e-3
    )
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col=9              
)


model = GoggleModel(
    ds_name="shuttle",
    input_dim=9
)

ckpt_path = ROOT / "tmp" / "shuttle.pt"

if ckpt_path.exists():
    model.model.load_state_dict(
        torch.load(ckpt_path, map_location=model.device)
    )
    print("Loaded checkpoint:", ckpt_path)
else:
    print("No checkpoint found — using in-memory trained model")

model.model.eval()


x_test = pd.read_csv(SAMPLE_DIR / "x_test.csv")
y_test = pd.read_csv(SAMPLE_DIR / "y_test.csv")


x_synth = model.sample(x_test)

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_test.iloc[:len(x_synth)].to_csv(
    SYNTH_DIR / "y_synth.csv", index=False
)

print("Synthetic data generated and saved successfully")


tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

raw_results = tstr.evaluate()

results = {
    model_name: {
        "Accuracy": scores["Accuracy"],
        "F1": scores["F1"]
    }
    for model_name, scores in raw_results.items()
}

print("\nTSTR Evaluation Results (SHUTTLE – STANDARD):\n")
for model_name, scores in results.items():
    print(model_name)
    for k, v in scores.items():
        print(f"  {k}: {v:.4f}")

print("\nTotal runtime (minutes):", round((time.time() - t0) / 60, 2))


Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
Training GOGGLE on (46400, 9)
 No checkpoint found — using last epoch weights.
No checkpoint found — using in-memory trained model
Synthetic data generated and saved successfully

Results saved to: Results\shuttle\goggle_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.1054
F1: 0.0292
AUC: 0.4215

MLP:
Accuracy: 0.7859
F1: 0.1257
AUC: 0.4072

RF:
Accuracy: 0.7859
F1: 0.1257
AUC: 0.5314

XGBoost:
Accuracy: 0.7859
F1: 0.1257
AUC: 0.5426

TSTR Evaluation Results (SHUTTLE – 